In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

sys.path.append(str(Path.cwd().parent))

from src.evaluation import evaluate

In [2]:
X_train, X_val = [
    pd.read_csv(f"../data/processed/X_{split}.csv") for split in ["train", "val"]
]
y_train, y_val = [
    pd.read_csv(f"../data/processed/y_{split}.csv").squeeze("columns")
    for split in ["train", "val"]
]

In [3]:
model = IsolationForest(
    n_estimators=500,
    max_samples=len(X_train),
    max_features=0.5,
    contamination="auto",
    random_state=42,
)

model.fit(X_train)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",500
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",198277
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",0.5
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",'auto'
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [4]:
anomaly_score = -model.score_samples(X_val)

y_val_np = y_val.values
anomaly_score_class_0 = anomaly_score[y_val_np == 0]
anomaly_score_class_1 = anomaly_score[y_val_np == 1]

print(pd.Series(anomaly_score_class_0).describe())
print(pd.Series(anomaly_score_class_1).describe())

count    42488.000000
mean         0.352714
std          0.020353
min          0.330604
25%          0.341706
50%          0.348172
75%          0.357767
max          0.799527
dtype: float64
count    236.000000
mean       0.545820
std        0.127538
min        0.340876
25%        0.456694
50%        0.536446
75%        0.650747
max        0.789536
dtype: float64


In [5]:
evaluate(model, X_val, y_val)

0.5500347666822977

In [6]:
output_dir = Path("../models")
output_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, "../models/isolation_forest.joblib")

['../models/isolation_forest.joblib']

With Feature Selection

In [7]:
cols_with_high_corr = ["V17", "V14", "V12", "V10"]

f_model = IsolationForest(
    n_estimators=500,
    max_samples=len(X_train),
    max_features=0.5,
    random_state=42,
    contamination="auto",
)
f_model.fit(X_train[cols_with_high_corr])

evaluate(f_model, X_val[cols_with_high_corr], y_val)

0.7716926106897755